# Plan v4 on TIDE (M1 resolution, M2 E4, M3 E4b, M4 E5/E6 + U-Net)

**Spawn:** PyTorch Notebook, 1 GPU, 8 CPU, 32 GB RAM. Plan of record: `agent_plan.md`. NCUGD is no longer evaluated.

**Track A** = YOLO-seg (E1/E2/E4…). **Track B** = U-Net (`train_semseg`, E3) for architecture comparison — do **not** launch E3 with `train_yolo`.

Order (one GPU job at a time; each new YOLO recipe must beat E1 threshold-free before the next builds on it):
1. Pull + setup (first time: upload `extras.zip`, ~2.5 GB, to `~/glass_crack/`, or set `EXTRAS_URL`)
2. One-time: archive the old E4 run (scale/colour only) as `E4old`
3. **M1** resolution ablation R0-R2 on frozen E1_s0 (GPU, no training)
4. **Track B** finish U-Net seeds (E3) if comparing architectures
5. **M2** E4 = E1 + scale/colour/motion blur/JPEG, seeds 0-2 (~2.5 h each)
6. **M3** E4b = E4 + 30% hard negatives, seeds 0-2
7. **M4** E5: CUBIT pretrain -> fine-tune; E6: CarDD pretrain -> fine-tune (only if E5 does not help)
8. Threshold-free comparison of everything + YOLO vs U-Net table

## 1. Pull and setup

In [ ]:
%%bash
cd ~/glass_crack/cracked && git pull --ff-only && git log -1 --oneline
# export EXTRAS_URL="https://drive.google.com/file/d/<id>/view"   # only if extras.zip is not uploaded
bash ~/glass_crack/cracked/scripts/tide/setup.sh 2>&1 | tail -20
ls ~/glass_crack/bundle/extras 2>/dev/null

## 2. One-time: archive the old E4 (scale/colour only) as E4old
The E4 config now also has motion blur + JPEG; without this the new E4_s0 would report "already finished".

In [ ]:
%%bash
source ~/glass_crack/cracked/scripts/tide/env.sh && cd "$CODE_DIR"
for d in runs evals; do
  [ -d "$OUT_ROOT/$d/E4_s0" ] && [ ! -d "$OUT_ROOT/$d/E4old_s0" ] && mv "$OUT_ROOT/$d/E4_s0" "$OUT_ROOT/$d/E4old_s0" && echo "moved $d/E4_s0 -> E4old_s0"
done
[ -f "$OUT_ROOT/logs/E4_s0.log" ] && mv "$OUT_ROOT/logs/E4_s0.log" "$OUT_ROOT/logs/E4old_s0.log" && rm -f "$OUT_ROOT/logs/E4_s0.pid"
sed -i 's/"exp_id": "E4"/"exp_id": "E4old"/' "$OUT_ROOT"/evals/E4old_s0/*.json  # compare_runs groups by this
sed -i 's/^E4,/E4old,/' "$OUT_ROOT/results/results.csv" && grep -c '^E4old,' "$OUT_ROOT/results/results.csv"

## 3. M1: resolution ablation on frozen E1_s0 (GPU, ~5-15 min per variant)
Same checkpoint, threshold re-frozen on valid per variant. R0 whole @640, R1 640 px tiles over the image at long side 1280, R2 whole @1024 / @1280.

In [ ]:
%%bash
source ~/glass_crack/cracked/scripts/tide/env.sh && cd "$CODE_DIR"
CK="$OUT_ROOT/runs/E1_s0/weights/best.pt"
run() { "$PY" -m glass_crack_models.eval.run_eval --checkpoint "$CK" --seed 0 --device 0 --no-map \
          --task yolo26s-seg --notes "M1 $*" --compute-units "TIDE $GPU_NAME" "$@" 2>&1 | tail -4; }
run --exp-id R0 --imgsz 640 --whole
run --exp-id R1 --imgsz 1280
run --exp-id R2_1024 --imgsz 1024 --whole
run --exp-id R2_1280 --imgsz 1280 --whole
"$PY" -m glass_crack_models.eval.compare_runs --runs E1_s0 R0_s0 R1_s0 R2_1024_s0 R2_1280_s0 --out "$OUT_ROOT/results/m1"
cat "$OUT_ROOT/results/m1/compare_runs.md"

## 4-6. Training jobs (background; survive closed tabs, not a server stop)
Set `CFG` / `NAME` / `SEED` / `MODULE`, run, then monitor below. Rerunning the same command resumes.

| Step | CFG | NAME | MODULE | Seeds |
|---|---|---|---|---|
| Track B | `e3_unet_r34.yaml` | `E3` | `train_semseg` | 0, 1, 2 |
| M2 | `e4_yolo_robust.yaml` | `E4` | `train_yolo` | 0, 1, 2 |
| M3 | `e4b_yolo_hardneg.yaml` | `E4b` | `train_yolo` | 0, 1, 2 |
| M4 A | `e5_pre_cubit.yaml` then `e5_yolo_cubit_ft.yaml` | `E5pre` / `E5` | `train_yolo` | 0; then 0–2 |
| M4 B | `e6_pre_cardd.yaml` then `e6_yolo_cardd_ft.yaml` | `E6pre` / `E6` | `train_yolo` | 0; then 0–2 |

U-Net uses `--device cuda` (string). YOLO uses `--device 0`. E8 (audited labels) waits for the manual crack/scratch audit.

In [ ]:
%%bash
source ~/glass_crack/cracked/scripts/tide/env.sh
# YOLO (Track A) example — default for M2+:
CFG=e4_yolo_robust.yaml; NAME=E4; SEED=0; MODULE=glass_crack_models.train.train_yolo; DEV=0
# U-Net (Track B) — uncomment to train/compare E3 (do NOT use train_yolo for this):
# CFG=e3_unet_r34.yaml; NAME=E3; SEED=0; MODULE=glass_crack_models.train.train_semseg; DEV=cuda
bash "$CODE_DIR/scripts/tide/job.sh" start ${NAME}_s$SEED $MODULE \
    --config $CFG --seed $SEED --device $DEV --workers "$WORKERS" --compute-units "TIDE $GPU_NAME"
bash "$CODE_DIR/scripts/tide/job.sh" status

In [ ]:
%%bash
# Monitor / stop. Change the job name as needed.
J=~/glass_crack/cracked/scripts/tide/job.sh
bash "$J" status
echo
bash "$J" tail E4_s0 30
# bash "$J" stop E4_s0

## 7. Threshold-free comparison and results table (CPU)
Includes YOLO vs U-Net when E3 seeds are present. Headline gate for stacking remains E1 (YOLO); U-Net is the architecture check.

In [ ]:
%%bash
source ~/glass_crack/cracked/scripts/tide/env.sh && cd "$CODE_DIR"
"$PY" -m glass_crack_models.eval.compare_runs
cat "$OUT_ROOT/results/compare_runs.md"
echo
echo "=== YOLO (E1) vs U-Net (E3) ==="
"$PY" -m glass_crack_models.eval.compare_runs --runs E1_s0 E1_s1 E1_s2 E3_s0 E3_s1 E3_s2 --out "$OUT_ROOT/results/yolo_vs_unet" 2>/dev/null \
  || "$PY" -m glass_crack_models.eval.compare_runs --runs E1_s0 E1_s1 E1_s2 E3_s0 --out "$OUT_ROOT/results/yolo_vs_unet"
cat "$OUT_ROOT/results/yolo_vs_unet/compare_runs.md"

In [ ]:
import os
import pandas as pd

p = os.path.expanduser("~/glass_crack/outputs/results/results.csv")
df = pd.read_csv(p).drop_duplicates(["exp_id", "seed", "checkpoint_sha256"], keep="last")
cols = ["exp_id", "seed", "threshold", "test_recall", "test_precision", "test_fp100", "test_mask_iou",
        "agdd_images_recall", "agdd_images_crack_recall", "agdd_image_recall", "agdd_image_crack_recall", "notes"]
display(df[[c for c in cols if c in df.columns]].sort_values(["exp_id", "seed"]))